# Evaluation report

Presents the output of `analysis/eval_steps.py` for one or more runs side by side. This notebook only reads the analysis output (`summary.json` and the curve CSVs); it computes nothing from the capture files.

Needs `matplotlib` and a Jupyter kernel (`ipykernel`). One color per run, one line style per head, in every figure. Where a run coincides with the start of a longer one (for example its first K steps), the shorter run is drawn on top.

## Configuration

`RUNS` maps a label to an analysis output directory (relative to the repository root, or absolute). `(directory, K)` selects the "first K steps" block written by `eval_steps.py --truncate-steps K`.

In [ ]:
RUNS = {
    "16 steps": ("checkpoints/sudoku_aux_linear_eval32/analysis", 16),
    "32 steps": "checkpoints/sudoku_aux_linear_eval32/analysis",
}
HEADS = ["q_head"]  # detached_linear is identical to q_head
SET_NAME = "all"
LOG_HISTOGRAMS = True

In [ ]:
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "analysis" / "report.py").exists())
sys.path.insert(0, str(ROOT))

%matplotlib inline
from analysis import report

runs = report.load_runs(RUNS, set_name=SET_NAME, root=ROOT)

## 1. Comparison

In [ ]:
report.summary_table(runs)

In [ ]:
report.operating_points_table(runs, HEADS)

## 2. Accuracy by step

A share of unsolved puzzles that keeps falling at the last steps means the model is still making progress there. The solving rate is the share of the puzzles unsolved at the previous step that are solved at this one.

In [ ]:
report.plot_accuracy_by_step(runs);

## 3. Operational curves

Each curve is traced by varying the halting threshold. The dotted line is the baseline where every puzzle stops at the same step.

In [ ]:
report.plot_operational_curves(runs, HEADS);

## 4. Stuck versus moving

Among the puzzles still unsolved at the previous step: how many keep the same predicted grid, and how much the states still move.

In [ ]:
report.plot_stuck_vs_moving(runs);

## 5. Halting steps at the selected thresholds

The hatched bar is the last step: puzzles that halt there or never reach the threshold.

In [ ]:
report.plot_halting_steps(runs, HEADS, log_y=LOG_HISTOGRAMS);

## 6. Flips

In [ ]:
report.flip_table(runs)

## 7. Repeated grids and cycles

For the puzzles unsolved at the last step, from the hash of the predicted grid at every step. A repeated grid does not prove that the latent state repeats, so a puzzle counts as frozen or cycling only if the pattern holds from the first repeated grid to the last step.

In [ ]:
report.cycle_table(runs)

In [ ]:
report.plot_unsolved_classes(runs);

In [ ]:
report.plot_cycle_histograms(runs, log_y=LOG_HISTOGRAMS);